# Welcome — you are logged in as *yourself*

This JupyterHub authenticated you through **Keycloak** (the same identity provider as Superset and Unity Catalog). Your notebook runs Spark **as you**: `uc_notebook` exchanges your Keycloak token for a **Unity Catalog** token and binds it to the Spark Connect session, so UC enforces RBAC for your principal.

There is **no admin default** — the compute edge holds zero ambient identity. You see exactly what you've been granted, nothing more.

In [1]:
import uc_notebook

spark = uc_notebook.uc_session()   # a Spark Connect session that is YOU
uc_notebook.whoami(spark)          # what can you see?

[analyst] visible schemas in analytics: ['gold']
[analyst] bronze: accessible
[analyst] silver: accessible
[analyst] gold  : accessible


## Read the gold layer (everyone with a persona can)

In [2]:
spark.sql("SELECT * FROM analytics.gold.customer_order_summary ORDER BY completed_orders DESC LIMIT 5").toPandas()

,customer_id,name,country,completed_orders,total_amount,last_order_ts
0,3,Chen Wei,SG,2,282.20,2026-01-09 16:44:00
1,4,Dana White,US,2,645.59,2026-01-12 21:30:00
2,1,Alice Nguyen,US,2,198.50,2026-01-10 12:15:00
3,2,Bruno Costa,BR,1,89.99,2026-01-04 11:45:00
4,5,Elif Demir,TR,0,0.00,NaT


## RBAC in action

If you logged in as **analyst**, the next cell is **denied by Unity Catalog** (analysts only get the gold layer). As **engineer** it succeeds. Same code, different identity — UC decides.

In [3]:
try:
    print(spark.sql("SELECT count(*) AS bronze_rows FROM analytics.bronze.orders").toPandas())
except Exception as e:
    print("DENIED by Unity Catalog (as expected for analyst):")
    print(str(e).splitlines()[0][:160])

DENIED by Unity Catalog (as expected for analyst):
(java.io.IOException) Failed to load table analytics.bronze.orders (HTTP 403): {"error_code":"PERMISSION_DENIED","details":[{"reason":"PERMISSION_DENIED","@type
